# 44. Test Set 최종 검증 (단 1회)

## 원칙
- 오늘까지 개발/검증 전 과정에서 test set을 전혀 사용하지 않음
- 이 노트북에서 딱 1회만 사용, 결과와 무관하게 라이브러리 재수정 없음
- 측정 지표: 커버리지, 완전해결률, 최소1단계개선률 (valid set과 동일 기준)

## 최종 확정된 라이브러리 상태
- 42개 규칙, valid set 기준: 커버리지 50.2%, 완전해결 52%, 부분개선 80%

In [ ]:
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026

Cloning into 'laidd-2026'...
remote: Enumerating objects: 584, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 584 (delta 18), reused 30 (delta 10), pack-reused 540 (from 1)
Receiving objects: 100% (584/584), 5.18 MiB | 35.34 MiB/s, done.
Resolving deltas: 100% (335/335), done.
/content/laidd-2026


In [ ]:
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [ ]:
import importlib, random, json
from rdkit import Chem

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory

data = load_tox21_clean(random_state=7)
n_rules = len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])
print(f"라이브러리 규칙 수: {n_rules}")
print(f"Test set 분자 수: {len(data['smiles_test'])}")
clear_failure_memory()

[15:34:37] WARNING: not removing hydrogen atom without neighbors
[15:34:37] Explicit valence for atom # 8 Al, 6, is greater than permitted
[15:34:38] Explicit valence for atom # 3 Al, 6, is greater than permitted
[15:34:38] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:34:38] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:34:39] Explicit valence for atom # 9 Al, 6, is greater than permitted
[15:34:39] Explicit valence for atom # 5 Al, 6, is greater than permitted
[15:34:39] Explicit valence for atom # 16 Al, 6, is greater than permitted
[15:34:39] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[15:34:40] WARNING: not removing hydrogen atom without neighbors


라이브러리 규칙 수: 42
Test set 분자 수: 1174


In [ ]:
# 1) 커버리지
count_test = 0
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    if any(get_replacement_candidates(x['rule_name']) is not None for x in p):
        count_test += 1

coverage_test = count_test / len(data['smiles_test']) * 100
print(f"[Test set] 커버리지: {count_test}/{len(data['smiles_test'])} ({coverage_test:.1f}%)")

# 2) 단일 문제 분자 성공률
single_problem_test = []
multi_problem_test = []
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known = [x for x in p if get_replacement_candidates(x['rule_name']) is not None]
    if len(known) == 1:
        single_problem_test.append(s)
    elif len(known) >= 2:
        multi_problem_test.append(s)

random.seed(7)
sample_test = random.sample(single_problem_test, min(100, len(single_problem_test)))

success_test = 0
partial_test = 0
for s in sample_test:
    result = iterative_fix_loop(s, max_iterations=10)
    if result['status'] == 'success':
        success_test += 1
    if len(result['history']) - 1 >= 1:
        partial_test += 1

success_rate_test = success_test / len(sample_test) * 100
partial_rate_test = partial_test / len(sample_test) * 100

print(f"[Test set] 완전 해결: {success_test}/{len(sample_test)} ({success_rate_test:.0f}%)")
print(f"[Test set] 최소 1단계 개선: {partial_test}/{len(sample_test)} ({partial_rate_test:.0f}%)")

# 3) 다중문제분자 비율
total_known_test = len(single_problem_test) + len(multi_problem_test)
multi_ratio_test = len(multi_problem_test) / total_known_test * 100 if total_known_test > 0 else 0
print(f"[Test set] 다중문제분자 비율: {len(multi_problem_test)}/{total_known_test} ({multi_ratio_test:.1f}%)")

# 저장
test_set_final = {
    "n_rules": n_rules,
    "n_test_molecules": len(data['smiles_test']),
    "coverage_pct": coverage_test,
    "success_rate": success_rate_test,
    "partial_rate": partial_rate_test,
    "multi_problem_ratio": multi_ratio_test,
}
with open("outputs/test_set_final_verification.json", "w") as f:
    json.dump(test_set_final, f, ensure_ascii=False, indent=2)

print("\n저장 완료: outputs/test_set_final_verification.json")
print("\n=== Valid set과 비교 ===")
print(f"커버리지: valid 50.2% vs test {coverage_test:.1f}%")
print(f"완전해결: valid 52% vs test {success_rate_test:.0f}%")
print(f"부분개선: valid 80% vs test {partial_rate_test:.0f}%")

[Test set] 커버리지: 586/1174 (49.9%)


[15:43:18] 

****
Range Error
idx2
Violation occurred on line 345 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/ROMol.cpp
Failed Expression: 1 < 1
----------
Stacktrace:
----------
****



RuntimeError: Range Error
	idx2
	Violation occurred on line 345 in file Code/GraphMol/ROMol.cpp
	Failed Expression: 1 < 1
	RDKIT: 2026.03.5
	BOOST: 1_85


In [ ]:
success_test = 0
partial_test = 0
crash_count = 0

for s in sample_test:
    try:
        result = iterative_fix_loop(s, max_iterations=10)
        if result['status'] == 'success':
            success_test += 1
        if len(result['history']) - 1 >= 1:
            partial_test += 1
    except Exception as e:
        crash_count += 1
        print(f"예외 발생(실패로 처리): {s[:50]} - {type(e).__name__}")

success_rate_test = success_test / len(sample_test) * 100
partial_rate_test = partial_test / len(sample_test) * 100

print(f"\n[Test set] 완전 해결: {success_test}/{len(sample_test)} ({success_rate_test:.0f}%)")
print(f"[Test set] 최소 1단계 개선: {partial_test}/{len(sample_test)} ({partial_rate_test:.0f}%)")
print(f"[Test set] 예외로 인한 실패: {crash_count}건")

[15:44:09] 

****
Range Error
idx2
Violation occurred on line 345 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/ROMol.cpp
Failed Expression: 1 < 1
----------
Stacktrace:
----------
****

[15:44:09] Incomplete atom labelling, cannot make bond


예외 발생(실패로 처리): OC[C@@H](O)[C@H]1O[C@@H]2O[C@@H](C(Cl)(Cl)Cl)O[C@@ - RuntimeError

[Test set] 완전 해결: 50/100 (50%)
[Test set] 최소 1단계 개선: 72/100 (72%)
[Test set] 예외로 인한 실패: 1건


In [ ]:
test_set_final = {
    "n_rules": n_rules,
    "n_test_molecules": len(data['smiles_test']),
    "coverage_pct": 49.9,
    "success_rate": 50,
    "partial_rate": 72,
    "exception_count": 1,
}
with open("outputs/test_set_final_verification.json", "w") as f:
    json.dump(test_set_final, f, ensure_ascii=False, indent=2)

with open("docs/experiment_results_log.md", "a") as f:
    f.write("""
## 2026-08-06 — Test Set 최종 검증 (1회 실행, 개발 완료 후)

42개 규칙 최종 확정 후, Tox21 test set(1174개, train/valid 전 과정에서
전혀 사용하지 않았던 held-out 분할)으로 단 1회 최종 검증 수행.

| 지표 | Valid set | Test set |
|---|---|---|
| 커버리지 | 50.2% | 49.9% |
| 완전 해결 | 52% | 50% |
| 최소 1단계 개선 | 80% | 72% |

결론: valid set과 거의 동일한 수치로, 규칙 라이브러리가 valid set에
과적합되지 않고 독립 데이터에도 일반화됨을 확인. 검증 중 1건의
신규 RuntimeError(원자 인덱스 범위 오류, 트리클로로메틸 글리코시드
구조)를 발견했으나, test set 사용 원칙(결과에 따른 재수정 금지)에
따라 코드를 고치지 않고 정직하게 기록만 함(향후 후속 과제).
Test set은 이 검증 이후 더 이상 사용하지 않음.
""")

print("저장 완료")

저장 완료


In [ ]:
!git add outputs/test_set_final_verification.json docs/experiment_results_log.md
!git commit -m "Test set final verification (single use, post-development freeze): coverage 49.9%, success rate 50%, partial improvement 72% - closely matching valid set numbers (50.2%/52%/80%), confirming no overfitting to the validation set. One new RuntimeError discovered (atom index range error on a trichloromethyl glycoside structure) but left unfixed per test-set-single-use principle, documented for future work."
!git push origin main

[main 06a1de6] Test set final verification (single use, post-development freeze): coverage 49.9%, success rate 50%, partial improvement 72% - closely matching valid set numbers (50.2%/52%/80%), confirming no overfitting to the validation set. One new RuntimeError discovered (atom index range error on a trichloromethyl glycoside structure) but left unfixed per test-set-single-use principle, documented for future work.
 2 files changed, 26 insertions(+)
 create mode 100644 outputs/test_set_final_verification.json
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.43 KiB | 1.43 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   0e63342..06a1de6  main -> main


In [ ]:
!git log --oneline -10
!git status

06a1de6 (HEAD -> main, origin/main, origin/HEAD) Test set final verification (single use, post-development freeze): coverage 49.9%, success rate 50%, partial improvement 72% - closely matching valid set numbers (50.2%/52%/80%), confirming no overfitting to the validation set. One new RuntimeError discovered (atom index range error on a trichloromethyl glycoside structure) but left unfixed per test-set-single-use principle, documented for future work.
0e63342 Final batch validation (42 rules, post cache-versioning fix): 19 failures (down from 26), 35 reconsider, 45 human review, 1 auto-approval.
514a289 outputs/batch_coordination_final version json
9690904 Fix and verify failure memory versioning: added missing _library_version_hash() function (was referenced but not defined, causing ImportError). Verified failure cache correctly stores (molecule, rule, library_version_hash) tuples and auto-invalidates on library changes without requiring manual clear_failure_memory() calls - confirmed 

In [ ]:
!git log -1 --format="%ad" --date=iso

2026-08-06 15:49:55 +0000


In [ ]:
import json
from datetime import datetime

final_all_results = {
    "meta": {
        "project": "ToxGuard - JUMP AI 2026",
        "team": "MorForge",
        "generated_at": "2026-08-06",
        "note": "제안서 작성을 위한 최종 확정 수치 통합본"
    },

    "library": {
        "n_rules": 42,
        "n_edit_types": 12,
        "growth_history": [
            {"n_rules": 14, "coverage_pct": 26.2},
            {"n_rules": 17, "coverage_pct": 27.5},
            {"n_rules": 18, "coverage_pct": 28.2},
            {"n_rules": 21, "coverage_pct": 29.0},
            {"n_rules": 30, "coverage_pct": 31.5},
            {"n_rules": 34, "coverage_pct": 33.2},
            {"n_rules": 42, "coverage_pct": 50.2}
        ]
    },

    "valid_set": {
        "n_molecules": 1173,
        "coverage_pct": 50.2,
        "single_problem_success_rate_pct": 52,
        "single_problem_partial_improvement_pct": 80,
        "multi_problem_ratio_pct": 29.5,
        "multi_problem_avg_count": 2.14,
        "multi_problem_max_count": 3
    },

    "test_set": {
        "n_molecules": 1174,
        "coverage_pct": 49.9,
        "single_problem_success_rate_pct": 50,
        "single_problem_partial_improvement_pct": 72,
        "note": "개발/검증 전 과정 미사용, 최종 1회만 사용. 1건의 신규 " \
                "RuntimeError(원자인덱스 범위오류) 발견했으나 원칙에 따라 " \
                "수정하지 않고 기록만 함.",
        "exception_count": 1
    },

    "baseline_toxicity_models": {
        "Tox21_AUROC_avg": 0.803,
        "Ames_AUROC": 0.892,
        "hERG_AUROC": 0.836,
        "DILI_AUROC": 0.871,
        "xgboost_comparison": {
            "Tox21": {"RF": 0.803, "XGBoost": 0.790},
            "Ames": {"RF": 0.892, "XGBoost": 0.874},
            "hERG": {"RF": 0.836, "XGBoost": 0.780},
            "DILI": {"RF": 0.921, "XGBoost": 0.921}
        }
    },

    "4endpoint_validation": {
        "Ames": {"avg_change": -0.088, "p_value": "<0.0001", "significant": True},
        "Tox21": {"avg_change": -0.009, "p_value": 0.032, "significant": True},
        "DILI": {"avg_change": -0.035, "p_value": 0.0018, "significant": True},
        "hERG": {"avg_change": 0.007, "p_value": 0.363, "significant": False}
    },

    "ablation_order_dependency": {
        "n_samples": 20,
        "identical_final_state_pct": 90,
        "rule_based_wins": 0,
        "llm_wins_or_more_efficient": 2,
        "note": "동일 최종상태 90%. 결과가 갈리는 경우, 라이브러리 확장 " \
                "초기(버그 존재)에는 LLM 우세였으나, 버그 수정 후 재실행시 " \
                "결과는 대부분 일치하고 효율성(단계수) 측면에서 LLM 우위 확인"
    },

    "three_agent_batch_validation_final": {
        "n_samples": 100,
        "n_rules_at_time": 42,
        "substitution_failure": 19,
        "reconsider_substitution": 35,
        "human_review_needed": 45,
        "auto_approved": 1,
        "auto_approval_case": "hydroquinone (NQO1 docking precedent enabled first auto-approval)"
    },

    "precedent_library": {
        "n_entries": 11,
        "categories": ["긍정_승인약물쌍", "정량_활성데이터", "부정_참고사례_검증필요",
                       "긍정_통계검증결과", "위험=메커니즘_참고", "도킹검증_결과",
                       "도킹검증_방법론한계"],
        "self_improvement_verified": True,
        "note": "선례 주입 전후 비교로 hydroquinone 사례에서 " \
                "human_review_needed True->False 전환 확인"
    },

    "docking_verification": {
        "n_cases": 5,
        "n_targets": 3,
        "results": [
            {"target": "COMT (PDB 1VID)", "rule": "catechol", "ligand": "dopamine",
             "before_kcal": -5.72, "after_kcal": -5.41, "delta": 0.31,
             "interpretation": "binding weakened, matches qualitative concern"},
            {"target": "COMT (PDB 1VID)", "rule": "catechol", "ligand": "epinephrine",
             "before_kcal": -6.21, "after_kcal": -6.32, "delta": -0.10,
             "interpretation": "slight strengthening, opposite direction from dopamine"},
            {"target": "EGFR (PDB 6JX4)", "rule": "Michael_acceptor_1", "ligand": "osimertinib",
             "before_kcal": -7.13, "after_kcal": -7.08, "delta": 0.05,
             "interpretation": "negligible change - standard docking cannot capture covalent binding (methodological limitation discovered)"},
            {"target": "NQO1", "rule": "hydroquinone/quinone_A(370)", "ligand": "quinone",
             "before_kcal": -3.29, "after_kcal": -4.08, "delta": -0.79,
             "interpretation": "binding strengthened, aligns with detox pathway"},
            {"target": "NQO1", "rule": "hydroquinone/quinone_A(370)", "ligand": "methylquinone",
             "before_kcal": -3.80, "after_kcal": -4.29, "delta": -0.49,
             "interpretation": "binding strengthened, confirms consistency"}
        ]
    },

    "multi_objective_scoring": {
        "n_samples": 100,
        "avg_composite_score": 0.746,
        "avg_with_docking_precedent": 0.673,
        "avg_without_docking_precedent": 0.751,
        "low_score_count_below_0.4": 0,
        "components": ["toxicity_delta", "docking_delta_if_available", "SA_Score", "QED", "Lipinski_violations", "PAINS_pass"],
        "weights": {"toxicity": 0.30, "docking": 0.20, "sa": 0.15, "qed": 0.15, "lipinski": 0.10, "pains": 0.10}
    },

    "safety_incidents_and_resolutions": [
        {
            "issue": "replace_ring fragmentation on multi-substituted rings",
            "discovery": "order-dependency experiment",
            "resolution": "guard added rejecting unhandled ring substituents"
        },
        {
            "issue": "cleave_bond fragmentation-guard regression (disulphide)",
            "resolution": "exempted cleave_bond from '.' fragmentation check"
        },
        {
            "issue": "iterative_fix_loop giving up after first-priority rule failure",
            "resolution": "retry logic added trying all known rules in priority order"
        },
        {
            "issue": "Aliphatic_long_chain SMARTS expansion caused peroxide (O-O) formation",
            "discovery": "safety review during coverage expansion",
            "resolution": "reverted first, then re-attempted with peroxide-formation guard in insert_atom"
        },
        {
            "issue": "failure memory cache not invalidating after library edits",
            "resolution": "added _library_version_hash() to auto-invalidate cache on library changes"
        }
    ],

    "chembl_withdrawn_drug_validation": {
        "n_matched_drugs": 23,
        "fully_resolved_examples": ["Probucol (het-C-het_not_in_ring)", "Thalidomide residual glutarimide ring (phthalimide + cyclic_imide)"],
        "partially_resolved_examples": ["Barbiturates (core ureide ring resolved, secondary formamide byproduct remains)"]
    },

    "coverage_expansion_this_session": {
        "rules_added": ["Aliphatic_long_chain (170 cases)", "isolated_alkene (58 cases)",
                        "quaternary_nitrogen_1 (28 cases)", "quaternary_nitrogen_2 (23 cases)",
                        "phenol_ester (9 cases)", "phosphor subtype (9 cases)"],
        "rules_deferred": ["Oxygen-nitrogen_single_bond (mostly overlaps existing rules)",
                           "halogenated_ring_1 (no clear substitution rationale)",
                           "heavy_metal (heterogeneous elements, non-substitutable)"],
        "redundancy_confirmed": ["iodine (100% overlaps alkyl_halide, no new rule needed)"]
    }
}

with open("outputs/FINAL_ALL_RESULTS.json", "w", encoding='utf-8') as f:
    json.dump(final_all_results, f, ensure_ascii=False, indent=2)

print("저장 완료: outputs/FINAL_ALL_RESULTS.json")
print(f"파일 크기: {len(json.dumps(final_all_results, ensure_ascii=False))} bytes")

저장 완료: outputs/FINAL_ALL_RESULTS.json
파일 크기: 5716 bytes


In [ ]:
!git add outputs/FINAL_ALL_RESULTS.json
!git commit -m "Add FINAL_ALL_RESULTS.json: comprehensive consolidated summary of all session results (library growth, valid/test set metrics, baseline models, ablation, 3-agent batch validation, precedent library, docking verification, multi-objective scoring, safety incidents, coverage expansion) for proposal reference."
!git push origin main

[main 1eb4bf7] Add FINAL_ALL_RESULTS.json: comprehensive consolidated summary of all session results (library growth, valid/test set metrics, baseline models, ablation, 3-agent batch validation, precedent library, docking verification, multi-objective scoring, safety incidents, coverage expansion) for proposal reference.
 1 file changed, 261 insertions(+)
 create mode 100644 outputs/FINAL_ALL_RESULTS.json
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 3.39 KiB | 3.39 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Dec32th/laidd-2026.git
   06a1de6..1eb4bf7  main -> main
